# 🖥️ Aula 04 — LLM Local no Colab (Ollama + Qwen 3.5 4B)
## Guilda de IA — Introdução à IA Generativa

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/aula04_ollama_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Rodando um modelo de linguagem **localmente** no Colab — sem API key, sem cloud.

---


## 1. Setup

⚠️ Vá em `Runtime → Change runtime type` → **T4 GPU**

In [ ]:
!nvidia-smi
!apt-get install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

In [ ]:
import subprocess, time, requests, json

subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(1)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env={**os.environ})

for i in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            break
    except:
        time.sleep(1)

!ollama pull qwen3.5:4b

## 2. Conversar com o modelo

⚠️ A primeira inferência demora ~2-3 min (warm up). Depois: ~80 tok/s.

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen3.5:4b"

def chat(messages, stream=False):
    payload = {"model": MODEL, "messages": messages, "stream": stream, "keep_alive": -1}
    r = requests.post(OLLAMA_URL, json=payload, timeout=300)
    if r.status_code != 200:
        raise Exception(f"Erro {r.status_code}: {r.text}")
    return r if stream else r.json()

# Warm up
print("🔥 Warm up...")
start = time.time()
r = chat([{"role": "user", "content": "Oi"}])
print(f"✅ Pronto em {time.time()-start:.1f}s")

In [ ]:
# Chat simples
r = chat([
    {"role": "system", "content": "Responda em português."},
    {"role": "user", "content": "Qual é a capital de Minas Gerais?"}
])
print(f"🤖 {r['message']['content']}")

In [ ]:
# Chat com memória
historico = [{"role": "system", "content": "Responda em português. Seja conciso."}]

historico.append({"role": "user", "content": "Meu nome é Lucas e eu ensino IA."})
r1 = chat(historico)
historico.append({"role": "assistant", "content": r1["message"]["content"]})
print(f"🤖: {r1['message']['content']}")

historico.append({"role": "user", "content": "Qual é o meu nome?"})
r2 = chat(historico)
historico.append({"role": "assistant", "content": r2["message"]["content"]})
print(f"🤖: {r2['message']['content']}")

## 3. API compatível com OpenAI

O endpoint `/v1/chat/completions` funciona como a API da OpenAI. Mesmo código, só muda a URL.

In [ ]:
r = requests.post("http://localhost:11434/v1/chat/completions", json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "O que é Python em uma frase?"}],
    "temperature": 0.7,
    "max_tokens": 64,
    "keep_alive": -1
}, timeout=300)
data = r.json()
print(f"🤖 {data['choices'][0]['message']['content']}")

---
✅ O padrão é o mesmo — Gemini, Ollama, OpenAI. Só muda a URL.

*Material da Guilda de IA — UFVJM 2026.1*